# Apparier deux sources, puis classer les paires

**Formation Big Data — ANSD / Data Innovation Lab**

Deux fichiers décrivent les mêmes personnes, sans identifiant commun : un
recensement et un registre d'état civil. Les noms y sont saisis différemment —
accents perdus, lettres doublées, inversions.

Nous allons les rapprocher, mesurer la qualité du résultat, puis entraîner un
modèle à départager les bonnes paires des mauvaises.

Un troisième fichier, la **vérité terrain**, indique les correspondances
réelles. Il ne sert **jamais** à apparier : uniquement à évaluer.

Le code est fourni. Exécutez, observez, modifiez les seuils pour explorer.

## 1. Session et données

In [ ]:
import os
import time
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

DONNEES = Path(os.environ.get("DONNEES", "/travail/donnees"))

spark = (
    SparkSession.builder
    .appName("appariement")
    .master("local[*]")             # local[2] si votre poste manque de mémoire
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

individus = spark.read.csv(str(DONNEES / "individus.csv"),
                           header=True, inferSchema=True)
actes = spark.read.csv(str(DONNEES / "etat_civil.csv"),
                       header=True, inferSchema=True)
verite = spark.read.csv(str(DONNEES / "etat_civil_verite.csv"),
                        header=True, inferSchema=True)

n_individus, n_actes, n_verite = individus.count(), actes.count(), verite.count()
print(f"Recensement  : {n_individus:,} individus".replace(",", " "))
print(f"État civil   : {n_actes:,} actes".replace(",", " "))
print(f"Vérité       : {n_verite:,} correspondances connues".replace(",", " "))

In [ ]:
import pyspark

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)

import sys
print(sys.executable)

#!pip uninstall pyspark
#!pip install pyspark==4.1.2

In [ ]:
individus.select("id_individu", "nom", "prenom", "sexe",
                 "date_naissance", "departement").show(4)
actes.select("id_acte", "nom_famille", "prenoms", "sexe",
             "date_naiss", "lieu_naissance").show(4)

Les mêmes personnes, mais des colonnes qui ne portent pas les mêmes noms et
des valeurs qui ne sont pas écrites de la même façon.

## 2. Normaliser

Avant toute comparaison, il faut ramener les valeurs à une forme commune :
casse, espaces, et surtout les dates, saisies ici dans trois formats
différents.

In [ ]:
def date_souple(colonne):
    """Convertit une date quel que soit son format de saisie.

    `try_to_date` renvoie une valeur manquante au lieu de lever une erreur —
    nécessaire depuis Spark 4, où le mode strict est actif par défaut.
    """
    return F.coalesce(
        F.try_to_date(colonne, "yyyy-MM-dd"),
        F.try_to_date(colonne, "dd/MM/yyyy"),
        F.try_to_date(colonne, "dd-MM-yyyy"),
    )


a = (individus
     .withColumn("nom_a", F.upper(F.trim("nom")))
     .withColumn("prenom_a", F.upper(F.trim("prenom")))
     .withColumn("naissance_a", date_souple(F.col("date_naissance")))
     .withColumn("lieu_a", F.upper(F.trim("departement")))
     .withColumnRenamed("sexe", "sexe_a")
     .select("id_individu", "nom_a", "prenom_a", "sexe_a",
             "naissance_a", "lieu_a"))

b = (actes
     .withColumn("nom_b", F.upper(F.trim("nom_famille")))
     .withColumn("prenom_b", F.upper(F.trim("prenoms")))
     .withColumn("naissance_b", date_souple(F.col("date_naiss")))
     .withColumn("lieu_b", F.upper(F.trim("lieu_naissance")))
     .withColumnRenamed("sexe", "sexe_b")
     .select("id_acte", "nom_b", "prenom_b", "sexe_b",
             "naissance_b", "lieu_b"))

a.show(3)
b.show(3)

## 3. Pourquoi on ne compare pas tout avec tout

Comparer chaque individu à chaque acte, c'est un **produit cartésien**.
Calculons son ampleur — sans l'exécuter.

In [ ]:
paires_possibles = n_individus * n_actes
print(f"Paires à comparer : {paires_possibles:,}".replace(",", " "))

# À raison d'un million de comparaisons par seconde, machine généreuse :
secondes = paires_possibles / 1e6
print(f"Soit environ {secondes / 3600:,.1f} heures de calcul"
      .replace(",", " "))
print("\nEt cela pour deux fichiers qui ne font que quelques centaines de "
      "milliers de lignes.")

Sur les volumes réels d'un institut de statistique, cette approche est
définitivement hors de portée. Il faut donc **ne comparer que ce qui a une
chance de correspondre**.

## 4. Le blocage

L'idée : construire une **clé de blocage** grossière, et ne comparer que les
enregistrements qui la partagent.

Notre clé combine deux éléments :

- `soundex` du nom — un code phonétique, identique pour les variantes
  d'orthographe ;
- l'année de naissance.

In [ ]:
# Soundex : le même code pour des orthographes différentes
exemples = spark.createDataFrame(
    [("DIOP", "DIOPP"), ("NDIAYE", "NDIAEY"), ("FALL", "FAL"),
     ("CISSÉ", "CISSE"), ("DIOP", "FALL")],
    ["ecriture_1", "ecriture_2"])

(exemples
 .withColumn("soundex_1", F.soundex("ecriture_1"))
 .withColumn("soundex_2", F.soundex("ecriture_2"))
 .withColumn("meme_bloc", F.soundex("ecriture_1") == F.soundex("ecriture_2"))
 .show())

In [ ]:
a = a.withColumn("bloc", F.concat_ws("_", F.soundex("nom_a"),
                                      F.year("naissance_a")))
b = b.withColumn("bloc", F.concat_ws("_", F.soundex("nom_b"),
                                      F.year("naissance_b")))

paires = a.join(b, on="bloc", how="inner")
n_paires = paires.count()

print(f"Paires candidates : {n_paires:,}".replace(",", " "))
print(f"Réduction : ×{paires_possibles / n_paires:,.0f}"
      .replace(",", " "))

Le blocage est **le** concept de l'appariement à grande échelle. Il a
cependant un défaut à connaître : une correspondance dont la clé diffère des
deux côtés — année de naissance erronée, par exemple — ne sera **jamais**
examinée. On perd du rappel dès cette étape.

En pratique, on emploie plusieurs clés de blocage complémentaires et on réunit
les résultats.

In [ ]:
# La taille des blocs conditionne le coût : un bloc de n lignes d'un côté et
# m de l'autre engendre n × m comparaisons.
(a.groupBy("bloc").count()
   .orderBy(F.col("count").desc())
   .show(5))

## 5. Mesurer la similarité

`levenshtein` compte le nombre de modifications nécessaires pour passer d'une
chaîne à l'autre. C'est une fonction native : aucune UDF, donc aucun surcoût.

In [ ]:
paires = (paires
          .withColumn("d_nom", F.levenshtein("nom_a", "nom_b"))
          .withColumn("d_prenom", F.levenshtein("prenom_a", "prenom_b"))
          .withColumn("meme_sexe", (F.col("sexe_a") == F.col("sexe_b")).cast("int"))
          .withColumn("meme_date",
                      (F.col("naissance_a") == F.col("naissance_b")).cast("int"))
          .withColumn("meme_lieu", (F.col("lieu_a") == F.col("lieu_b")).cast("int"))
          # Distance rapportée à la longueur : « DIOP/DIOPP » et
          # « NDIAYE/NDIAEY » n'ont pas la même gravité
          .withColumn("ratio_nom",
                      F.col("d_nom") / F.greatest(F.length("nom_a"),
                                                  F.length("nom_b")))
          .withColumn("ratio_prenom",
                      F.col("d_prenom") / F.greatest(F.length("prenom_a"),
                                                     F.length("prenom_b"))))

paires.select("nom_a", "nom_b", "d_nom", "ratio_nom",
              "prenom_a", "prenom_b", "d_prenom").show(8)

## 6. Une première règle, et son évaluation

Appliquons un seuil simple, puis confrontons le résultat à la vérité terrain.

In [ ]:

retenues = paires.filter(
    (F.col("d_nom") <= 2) & (F.col("d_prenom") <= 2) & (F.col("meme_sexe") == 1)
).cache()

n_retenues = retenues.count()
vrais_positifs = retenues.join(verite, on=["id_acte", "id_individu"]).count()

precision = 100 * vrais_positifs / n_retenues
rappel = 100 * vrais_positifs / n_verite

print(f"Paires retenues  : {n_retenues:,}".replace(",", " "))
print(f"Vrais positifs   : {vrais_positifs:,}".replace(",", " "))
print(f"\nPrécision : {precision:.1f} %  (parmi les paires retenues, "
      "combien sont justes)")
print(f"Rappel    : {rappel:.1f} %  (parmi les correspondances réelles, "
      "combien ont été trouvées)")

Voilà le compromis fondamental de l'appariement : **on trouve presque tout,
mais on retient beaucoup de faux**. Deux personnes d'une même famille, nées la
même année, avec des prénoms proches, se ressemblent trop pour être départagées
par ces seules variables.

In [ ]:
# Faire varier le seuil déplace le compromis, sans le supprimer
print(f"{'seuil':>6} {'retenues':>12} {'précision':>11} {'rappel':>9}")
for seuil in [1, 2, 3]:
    sel = paires.filter((F.col("d_nom") <= seuil)
                        & (F.col("d_prenom") <= seuil)
                        & (F.col("meme_sexe") == 1))
    n = sel.count()
    vp = sel.join(verite, on=["id_acte", "id_individu"]).count()
    print(f"{seuil:>6} {n:>12,} {100 * vp / max(n, 1):>10.1f} % "
          f"{100 * vp / n_verite:>8.1f} %".replace(",", " "))

> Le choix du seuil n'est pas une décision technique : elle dépend de l'usage.
> Pour un fichier qui sera vérifié à la main, on privilégie le rappel. Pour une
> publication automatique, la précision.

## 7. Faire mieux qu'un seuil : classer les paires

Nous disposons de paires candidates et, pour une partie d'entre elles, de la
réponse. C'est un problème d'**apprentissage supervisé** : entraîner un modèle
à reconnaître une bonne paire.

C'est ainsi que procèdent les instituts de statistique pour l'appariement de
registres.

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

retenues.unpersist()

# On garde un ensemble large de candidates, et on y attache l'étiquette connue
candidates = paires.filter((F.col("d_nom") <= 3) & (F.col("d_prenom") <= 4))

etiquetees_init = (candidates
              .join(verite.withColumn("vrai", F.lit(1)),
                    on=["id_acte", "id_individu"], how="left")
              .withColumn("vrai", F.coalesce(F.col("vrai"), F.lit(0))))

# Sampling 
vraies = (
    etiquetees_init
    .filter(F.col("vrai") == 1)
    .orderBy(F.rand(seed=42))
    .limit(10_000_000)
)

fausses = (
    etiquetees_init
    .filter(F.col("vrai") == 0)
    .orderBy(F.rand(seed=42))
    .limit(1_000_000)
)

etiquetees = vraies.unionByName(fausses).cache()

n_total = etiquetees.count()
n_positives = etiquetees.filter(F.col("vrai") == 1).count()
print(f"Paires étiquetées : {n_total:,}".replace(",", " "))
print(f"Dont vraies       : {n_positives:,} "
      f"({100 * n_positives / n_total:.1f} %)".replace(",", " "))

**Moins de 6 % de paires positives.** Retenez ce chiffre : il va poser un
problème.

### Préparer les variables

MLlib attend toutes les variables explicatives rassemblées dans une seule
colonne vectorielle. C'est le rôle de `VectorAssembler`.

In [ ]:
variables_faibles = ["d_nom", "d_prenom", "meme_sexe",
                     "ratio_nom", "ratio_prenom"]

assembleur = VectorAssembler(inputCols=variables_faibles, outputCol="features")

donnees = (assembleur.transform(etiquetees)
           .select("features", F.col("vrai").alias("label"))
           .cache())

apprentissage, test = donnees.randomSplit([0.7, 0.3], seed=42)
print(f"Apprentissage : {apprentissage.count():,}".replace(",", " "))
print(f"Test          : {test.count():,}".replace(",", " "))

### Premier essai — et le piège du déséquilibre

In [ ]:
modele = LogisticRegression(maxIter=20).fit(apprentissage)
predictions = modele.transform(test)

auc = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(predictions)

vp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 1)).count()
fp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 0)).count()
fn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 1)).count()

print(f"AUC       : {auc:.3f}")
print(f"Prédites positives : {vp + fp:,}".replace(",", " "))
print(f"Précision : {100 * vp / max(vp + fp, 1):.1f} %")
print(f"Rappel    : {100 * vp / max(vp + fn, 1):.1f} %")

Une AUC excellente, mais un modèle qui ne prédit **jamais** la classe
positive.

Il n'y a pas de contradiction : l'AUC mesure la capacité à **ordonner** les
paires, et de ce point de vue le modèle est bon. Mais avec très peu de positifs, la
stratégie qui minimise l'erreur est de tout classer en négatif — et le seuil de
décision par défaut, fixé à 0,5, n'est jamais atteint.

C'est le piège classique des données déséquilibrées, et il se corrige.

In [ ]:
# On donne davantage de poids aux exemples positifs, en proportion de leur rareté
ratio = n_positives / n_total
poids = (1 - ratio) / ratio
print(f"Poids attribué aux paires positives : ×{poids:.1f}")

donnees_ponderees = (assembleur.transform(etiquetees)
                     .select("features", F.col("vrai").alias("label"))
                     .withColumn("poids",
                                 F.when(F.col("label") == 1, F.lit(poids))
                                  .otherwise(F.lit(1.0)))
                     .cache())

apprentissage, test = donnees_ponderees.randomSplit([0.7, 0.3], seed=42)

modele = LogisticRegression(maxIter=20, weightCol="poids").fit(apprentissage)
predictions = modele.transform(test).cache()

auc = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(predictions)
vp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 1)).count()
fp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 0)).count()
fn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 1)).count()

print(f"\nAUC       : {auc:.3f}")
print(f"Précision : {100 * vp / max(vp + fp, 1):.1f} %")
print(f"Rappel    : {100 * vp / max(vp + fn, 1):.1f} %")

Le modèle fonctionne — mais il ne fait guère mieux que notre seuil manuel.

Ce n'est pas l'algorithme qui est en cause : ce sont les **variables**. Nous
n'avons donné au modèle que des distances entre noms et prénoms. Ajoutons ce
que tout appariement réel utilise : la date de naissance exacte et le lieu.

In [ ]:
variables_completes = variables_faibles + ["meme_date", "meme_lieu"]

assembleur = VectorAssembler(inputCols=variables_completes, outputCol="features")

donnees_completes = (assembleur.transform(etiquetees)
                     .select("features", F.col("vrai").alias("label"))
                     .withColumn("poids",
                                 F.when(F.col("label") == 1, F.lit(poids))
                                  .otherwise(F.lit(1.0)))
                     .cache())

apprentissage, test = donnees_completes.randomSplit([0.7, 0.3], seed=42)

modele = LogisticRegression(maxIter=20, weightCol="poids").fit(apprentissage)
predictions = modele.transform(test)

auc = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(predictions)
vp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 1)).count()
fp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 0)).count()
fn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 1)).count()

print(f"AUC       : {auc:.3f}")
print(f"Précision : {100 * vp / max(vp + fp, 1):.1f} %")
print(f"Rappel    : {100 * vp / max(vp + fn, 1):.1f} %")

**L'algorithme n'a pas changé.** Ni les données, ni la méthode
d'entraînement. Seules deux variables ont été ajoutées — et le résultat n'a plus
rien à voir.

C'est la leçon la plus transférable de ce bloc : dans un problème
d'apprentissage, **le choix des variables pèse plus lourd que le choix de
l'algorithme**. Un statisticien qui connaît son domaine construit de meilleures
variables qu'un modèle sophistiqué ne compense.

> Nuance à garder en tête : ici les dates de naissance sont exactes des deux
> côtés. Sur des registres réels, elles sont souvent approximatives ou
> manquantes, et le résultat serait moins net.

In [ ]:
# Le poids de chaque variable dans le modèle
for nom, coefficient in zip(variables_completes, modele.coefficients):
    print(f"  {nom:<15} {coefficient:>8.3f}")

## 8. Les doublons dans une même source

Le même raisonnement s'applique à l'intérieur d'un seul fichier : deux
enregistrements très proches sont probablement la même personne saisie deux
fois.

In [ ]:
doublons = (a.alias("g")
            .join(a.alias("d"),
                  (F.col("g.bloc") == F.col("d.bloc"))
                  & (F.col("g.id_individu") < F.col("d.id_individu")))
            .withColumn("d_nom", F.levenshtein("g.nom_a", "d.nom_a"))
            .withColumn("d_prenom", F.levenshtein("g.prenom_a", "d.prenom_a"))
            .filter((F.col("d_nom") <= 1) & (F.col("d_prenom") <= 1)
                    & (F.col("g.naissance_a") == F.col("d.naissance_a"))))

print(f"Doublons probables : {doublons.count():,}".replace(",", " "))
doublons.select("g.id_individu", "d.id_individu", "g.nom_a", "g.prenom_a",
                "g.naissance_a").show(5)

> La condition `id_individu <` évite de retenir deux fois la même paire, et
> de comparer un enregistrement avec lui-même.

## 9. Ce qu'il faut retenir

- Comparer toutes les paires est **impossible** au-delà de quelques milliers de
  lignes. Le **blocage** ramène le problème à une taille traitable — au prix
  d'un peu de rappel, car une correspondance dont la clé diffère est perdue
  d'emblée.
- `soundex` regroupe les variantes d'orthographe, `levenshtein` mesure l'écart.
  Ce sont des fonctions natives : pas d'UDF, pas de surcoût.
- Un seuil manuel donne un **fort rappel et une faible précision**. Le déplacer
  échange l'un contre l'autre ; le choix relève de l'usage, pas de la technique.
- La **vérité terrain** ne sert jamais à apparier, uniquement à évaluer.
- Sur des classes déséquilibrées, un modèle peut afficher une excellente AUC
  tout en ne prédisant jamais la classe rare. La **pondération** corrige ce
  comportement.
- Et surtout : **le choix des variables pèse plus lourd que le choix de
  l'algorithme**.

In [ ]:
spark.stop()
print("Session arrêtée.")